<a href="https://colab.research.google.com/github/santoshkothapalli/genai/blob/main/Langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langgraph langchain openai

In [ ]:
## Githb comment
from typing import TypedDict, List, Optional

class ShoppingState(TypedDict):
    query: str
    results: List[dict]
    best_option: Optional[dict]
    user_decision: Optional[str]

In [ ]:
def search_products(state: ShoppingState):

    query = state["query"]

    results = [
        {"platform": "Amazon", "price": 398, "url": "amazon.com/product"},
        {"platform": "Walmart", "price": 389, "url": "walmart.com/product"},
        {"platform": "BestBuy", "price": 399, "url": "bestbuy.com/product"}
    ]

    print("Search results gathered from ecommerce platforms")

    return {"results": results}

In [ ]:
def compare_prices(state: ShoppingState):

    results = state["results"]

    best = min(results, key=lambda x: x["price"])

    print("Best price identified:", best)

    return {"best_option": best}

In [ ]:
def human_review(state: ShoppingState):

    best = state["best_option"]

    print("\nBest option found:")
    print(best)

    decision = input(
        "Approve purchase? (yes / no / change): "
    )

    return {"user_decision": decision}

In [ ]:
def payment_agent(state: ShoppingState):

    decision = state["user_decision"]

    if decision.lower() == "yes":

        best = state["best_option"]

        print("\nProcessing payment...")
        print("Purchasing from:", best["platform"])
        print("Price:", best["price"])

        return {"status": "payment_completed"}

    else:
        print("Purchase cancelled or modified.")

        return {"status": "stopped"}

In [ ]:
from langgraph.graph import StateGraph

workflow = StateGraph(ShoppingState)

workflow.add_node("search", search_products)
workflow.add_node("compare", compare_prices)
workflow.add_node("human_review", human_review)
workflow.add_node("payment", payment_agent)

workflow.set_entry_point("search")

workflow.add_edge("search", "compare")
workflow.add_edge("compare", "human_review")
workflow.add_edge("human_review", "payment")

app = workflow.compile()

In [ ]:
state = {
    "query": "Sony WH-1000XM5 headphones",
    "results": [],
    "best_option": None,
    "user_decision": None
}

app.invoke(state)

Search results gathered from ecommerce platforms
Best price identified: {'platform': 'Walmart', 'price': 389, 'url': 'walmart.com/product'}

Best option found:
{'platform': 'Walmart', 'price': 389, 'url': 'walmart.com/product'}
Approve purchase? (yes / no / change): change
Purchase cancelled or modified.


{'query': 'Sony WH-1000XM5 headphones',
 'results': [{'platform': 'Amazon', 'price': 398, 'url': 'amazon.com/product'},
  {'platform': 'Walmart', 'price': 389, 'url': 'walmart.com/product'},
  {'platform': 'BestBuy', 'price': 399, 'url': 'bestbuy.com/product'}],
 'best_option': {'platform': 'Walmart',
  'price': 389,
  'url': 'walmart.com/product'},
 'user_decision': 'change'}